# Verification of Generated Texts — DeepSeek API — batch_13

Runs `prompt_verification.md` on every JSON file in `prompts_batches_20/batch_13/`.

- **System prompt**: `prompt_verification.md` (the auditor instructions)
- **User message**: the JSON object `{id, specie, specie_en, deb_model, prompt, generated_text}`
- **Result saved**: key `verification` added to each JSON file (skip if already present or if `generated_text` is missing)

In [1]:
import json
import os
import time
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("__file__").resolve().parents[2] / ".env") if "__file__" in dir() else load_dotenv()

# ── Configuration ─────────────────────────────────────────────────────────────
API_KEY            = os.environ.get("DEEPSEEK_API_KEY")
MODEL              = "deepseek-v4-pro"
API_URL            = "https://api.deepseek.com/chat/completions"
GENERATED_DIR      = Path("../prompts_batches_20/batch_13")
PROMPT_FILE        = Path("../verif_prompts/prompt_verification.md")

DELAY_BETWEEN_CALLS = 5
MAX_RETRIES         = 6
RETRY_DELAY         = 30
# ──────────────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError("DEEPSEEK_API_KEY environment variable is not set.")

SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")

json_files = sorted(GENERATED_DIR.glob("*.json"))
print(f"Model          : {MODEL}")
print(f"API URL        : {API_URL}")
print(f"Found          : {len(json_files)} file(s) in {GENERATED_DIR}")



Model          : deepseek-v4-pro
API URL        : https://api.deepseek.com/chat/completions
Found          : 370 file(s) in ../prompts_batches_20/batch_13


In [2]:
def call_deepseek_verify(system_prompt: str, user_content: str) -> tuple[str, dict]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"], data.get("usage", {})


# ── Main loop ─────────────────────────────────────────────────────────────────
fatal = False
for idx, filepath in enumerate(json_files, 1):
    data = json.loads(filepath.read_text(encoding="utf-8"))

    if data.get("verification") is not None:
        print(f"[SKIP-verified]  {filepath.name}")
        continue

    if not data.get("generated_text"):
        print(f"[SKIP-no-text]   {filepath.name}")
        continue

    # Build the user message: only the fields the auditor needs
    user_payload = json.dumps({
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "specie_en":      data.get("specie_en"),
        "deb_model":      data.get("deb_model"),
        "prompt":         data.get("prompt"),
        "generated_text": data.get("generated_text"),
    }, ensure_ascii=False)

    print(f"[{idx:04d}/{len(json_files)}] {filepath.name} ...", end=" ", flush=True)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw_result, usage = call_deepseek_verify(SYSTEM_PROMPT, user_payload)

            # Parse the JSON response from DeepSeek
            try:
                verification = json.loads(raw_result)
            except json.JSONDecodeError:
                # Sometimes the model wraps the JSON in ```json ... ```
                stripped = raw_result.strip().removeprefix("```json").removesuffix("```").strip()
                verification = json.loads(stripped)

            data["verification"] = verification
            filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

            verdict = verification.get("verdict", "?")
            critical = verification.get("critical_errors", "?")
            cached = usage.get("prompt_cache_hit_tokens", 0)
            print(f"{verdict} | {critical} critical | {usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out / {cached} cached")
            break

        except urllib.error.HTTPError as e:
            body_err = e.read().decode(errors="replace")
            print(f"\n  HTTP {e.code} on attempt {attempt}: {body_err[:200]}")
            if e.code == 401:
                print("  Invalid API key — check DEEPSEEK_API_KEY. Aborting.")
                fatal = True
                break
            elif e.code == 404:
                print(f"  Model '{MODEL}' not found. Aborting.")
                fatal = True
                break
            elif e.code == 403:
                print("  Access forbidden — skipping this file.")
                break
            elif e.code == 429:
                wait = RETRY_DELAY * (2 ** (attempt - 1))
                print(f"  Rate limited — waiting {wait}s before retry...", flush=True)
                time.sleep(wait)
            elif attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
        except Exception as e:
            print(f"\n  Error on attempt {attempt}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    else:
        print(f"  FAILED after {MAX_RETRIES} attempts — skipping.")

    if fatal:
        break

    if idx < len(json_files):
        time.sleep(DELAY_BETWEEN_CALLS)

print("\nAll done.")

[0001/370] 4441_Neoceratodus_forsteri_std.json ... FAIL | 8 critical | 38269 in / 3872 out / 12032 cached
[0002/370] 4442_Neochelon_falcipinnis_std.json ... FAIL | 5 critical | 33702 in / 2271 out / 12032 cached
[0003/370] 4443_Neochmia_temporalis_std.json ... PASS | 0 critical | 35013 in / 1459 out / 12032 cached
[0004/370] 4444_Neodraco_skottsbergi_abj.json ... FAIL | 9 critical | 34272 in / 3582 out / 12032 cached
[0005/370] 4445_Neofelis_nebulosa_stx.json ... FAIL | 12 critical | 35402 in / 3011 out / 12032 cached
[0006/370] 4446_Neofiber_alleni_stx.json ... FAIL | 11 critical | 34647 in / 3599 out / 12032 cached
[0007/370] 4447_Neogale_vison_stx.json ... FAIL | 3 critical | 35695 in / 1923 out / 12032 cached
[0008/370] 4448_Neogobius_caspius_std.json ... FAIL | 6 critical | 33077 in / 1738 out / 12032 cached
[0009/370] 4449_Neogobius_fluvialitis_std.json ... FAIL | 6 critical | 33186 in / 4404 out / 12032 cached
[0010/370] 4450_Neogobius_melanostomus_std.json ... 
  Error on attem

In [3]:
# ── Summary ───────────────────────────────────────────────────────────────────
from collections import Counter

counts = Counter()
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    if not data.get("generated_text"):
        counts["NO_GENERATED_TEXT"] += 1
        continue
    v = data.get("verification")
    if v is None:
        counts["NOT_VERIFIED"] += 1
    else:
        counts[v.get("verdict", "UNKNOWN")] += 1

total = sum(counts.values())
print(f"=== Verification Summary ({total} files) ===")
for verdict, count in counts.most_common():
    print(f"  {verdict}: {count}")

=== Verification Summary (370 files) ===
  FAIL: 336
  PASS: 34


In [4]:
# ── Detail: FAIL files ────────────────────────────────────────────────────────
fails = []
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification")
    if v and v.get("verdict") == "FAIL":
        fails.append({
            "file":            filepath.name,
            "specie":          data.get("specie"),
            "critical_errors": v.get("critical_errors"),
            "error_codes":     [e.get("code") for e in v.get("errors", [])],
        })

print(f"FAIL ({len(fails)} files):")
for r in fails:
    print(f"  {r['file']}  —  {r['specie']}  —  {r['critical_errors']} critical  —  {r['error_codes']}")

FAIL (336 files):
  4441_Neoceratodus_forsteri_std.json  —  Neoceratodus_forsteri  —  8 critical  —  ['G1c', 'E2', 'B3', 'B3', 'B4', 'F4', 'C1', 'C1', 'C1', 'D3', 'D2', 'H1', 'H7', 'H6', 'H3']
  4442_Neochelon_falcipinnis_std.json  —  Neochelon_falcipinnis  —  5 critical  —  ['E4', 'E2', 'E2', 'E2', 'B3', 'H2', 'H3']
  4444_Neodraco_skottsbergi_abj.json  —  Neodraco_skottsbergi  —  9 critical  —  ['A2', 'D1', 'E2', 'G1f', 'G1e', 'B3', 'G1a', 'H1', 'F5', 'H2', 'H7']
  4445_Neofelis_nebulosa_stx.json  —  Neofelis_nebulosa  —  12 critical  —  ['A4', 'C5', 'C5', 'E1', 'E2', 'E2', 'E2', 'D3', 'D3', 'D3', 'D3', 'D3', 'G1f', 'H6', 'B3']
  4446_Neofiber_alleni_stx.json  —  Neofiber_alleni  —  11 critical  —  ['E2', 'F5', 'D2', 'C7', 'E2', 'B4', 'G1f', 'C1', 'A2', 'H2']
  4447_Neogale_vison_stx.json  —  Neogale_vison  —  3 critical  —  ['E2', 'G1f', 'H5', 'H6', 'C5']
  4448_Neogobius_caspius_std.json  —  Neogobius_caspius  —  6 critical  —  ['E2', 'E2', 'E2', 'E2', 'E2', 'E6']
  4449_Neogobius_

In [5]:
# ── Detail: GENERATION_FAILURE files ─────────────────────────────────────────
failures = []
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification")
    if v and v.get("verdict") == "GENERATION_FAILURE":
        failures.append({"file": filepath.name, "specie": data.get("specie")})

print(f"GENERATION_FAILURE ({len(failures)} files):")
for r in failures:
    print(f"  {r['file']}  —  {r['specie']}")

GENERATION_FAILURE (0 files):
